# Experiment - Company encoding & history

> **Takeaway -** Engineer leakage-safe company *history* (prior bookings + prior cancel-rate) instead of feeding the raw company - it's the only company representation that beats ignoring company, and naive target-encoding (hence CatBoost) doesn't help.

> **Decided -> the `history` bundle.** On the temporal hold-out the engineered
> history beat every alternative (HistGB **AUC 0.9124 / AP 0.7748** vs `drop`
> 0.9102 / 0.7698), and the prior-cancel-rate quintiles run **0.086 -> 0.315** - a
> company's past cancellations clearly predict its next. Naive `target_oof` was
> *worse* than dropping company (11,660 companies, mostly singletons) - which is
> also why CatBoost wasn't worth adding. Implemented in 00 §5.2 as `has_company`,
> `is_repeat_company`, `company_prior_bookings`, `company_prior_cancel_rate`;
> logged in `reports/open_decisions.md`.


We use `company_name_clean` (clustered) rather than the raw `company_name_combined`
(dropped in §3.0.5) because clustering merges variants ("BMW AG" / "BMW" /
"B.M.W. GmbH" -> one entity), so frequency and history are counted correctly.

**Arms (each ADDS one company representation to the full feature set):**
- `drop` - baseline (no company feature; the pre-decision state).
- `has_company` - single binary (a company is attached at all).
- `is_repeat` - binary: this company has >=1 *prior* booking (time-aware).
- `freq_bucket` - company total booking count, bucketed + OHE (NB: full-frame
  frequency, so mildly leaky - flagged).
- `history` - **the chosen one**: has_company + log(prior_bookings) +
  prior_cancel_rate. **Strictly leakage-safe**: only bookings with
  `created` < this booking's `created` count.
- `target_oof` - out-of-fold smoothed target encoding of the company id.

Models: LogReg + HistGradientBoosting; metric ROC AUC + AP on the temporal
hold-out with a bootstrap 95% CI. Saved to
`reports/tables/00_audit/company_encoding_benchmark.csv`.

## What this experiment is about
About a third of bookings come from a company, and companies behave differently
from leisure guests - especially a company that has cancelled a lot before. We
test whether knowing the company and **its past behaviour** helps predict
cancellation, and how to represent 11,660 different companies usefully.

In [1]:
import sys
from pathlib import Path
_here = Path.cwd().resolve()
while not (_here / "pyproject.toml").exists() and _here != _here.parent:
    _here = _here.parent
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import warnings, numpy as np, pandas as pd
from src.data_loader import load_clean_reservations
from src.features import model_feature_roster
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, average_precision_score
warnings.filterwarnings("ignore", category=UserWarning)

TARGET_COL = "status"
COMPANY_COL = "company_name_clean"

def _obj(frame):
    if isinstance(frame, pd.Series): frame = frame.to_frame()
    return frame.astype("string").to_numpy(dtype=object, na_value=np.nan)

def target_encode_oof(train_cat, y_train, test_cat, *, n_splits=5, smoothing=20.0, seed=42):
    train_cat = train_cat.astype("string").fillna("__NA__").to_numpy()
    test_cat  = test_cat.astype("string").fillna("__NA__").to_numpy()
    y_train = np.asarray(y_train, float); gm = y_train.mean()
    def _fit(cats, ys):
        d = pd.DataFrame({"c":cats,"y":ys}); g=d.groupby("c")["y"].agg(["mean","count"])
        return ((g["count"]*g["mean"]+smoothing*gm)/(g["count"]+smoothing)).to_dict()
    oof = np.full(len(train_cat), gm)
    for tr,va in KFold(n_splits,shuffle=True,random_state=seed).split(train_cat):
        m=_fit(train_cat[tr],y_train[tr]); oof[va]=[m.get(x,gm) for x in train_cat[va]]
    full=_fit(train_cat,y_train)
    return oof.reshape(-1,1), np.array([full.get(x,gm) for x in test_cat]).reshape(-1,1)

def bootstrap_auc_ci(y,p,n_boot=400,seed=42):
    rng=np.random.default_rng(seed); y=np.asarray(y); p=np.asarray(p); n=len(y); s=[]
    for _ in range(n_boot):
        idx=rng.integers(0,n,n)
        if len(np.unique(y[idx]))<2: continue
        s.append(roc_auc_score(y[idx],p[idx]))
    return np.percentile(s,[2.5,97.5])

In [2]:
# ---- load + build company features (time-aware history is leakage-safe) ----
df = load_clean_reservations().dropna(subset=[TARGET_COL]).copy()
df["y"] = df[TARGET_COL].astype(int)
df["created"] = pd.to_datetime(df["created"], utc=True)

comp = df[COMPANY_COL].astype("string").fillna("").str.strip()
df["has_company"] = (comp.str.len() > 0).astype(int)

# full-frame frequency (mildly leaky -> used only by the freq_bucket arm)
freq = comp[comp.str.len() > 0].value_counts()
df["company_freq"] = comp.map(freq).fillna(0).astype(int)
df["freq_bucket"] = pd.cut(df["company_freq"], bins=[-1,0,1,5,20,10**9],
                           labels=["none","1","2-5","6-20","20+"]).astype("string")

# time-aware history: only bookings with created < this booking's created count.
d = df.sort_values("created", kind="mergesort").copy()
nonempty = d[COMPANY_COL].astype("string").fillna("").str.len() > 0
g = d.assign(_c=d[COMPANY_COL].astype("string")).groupby("_c")
d["company_prior_bookings"] = g.cumcount()
d["company_prior_cancels"]  = g["y"].cumsum() - d["y"]
d.loc[~nonempty, ["company_prior_bookings","company_prior_cancels"]] = 0
d["company_prior_cancel_rate"] = (d["company_prior_cancels"] /
                                  d["company_prior_bookings"].replace(0, np.nan))
df = d.sort_index()
df["is_repeat_company"] = ((df["has_company"]==1) & (df["company_prior_bookings"]>0)).astype(int)

NUMERIC_BASE, CATEGORICAL_BASE = model_feature_roster(df, exclude={
    "y","has_company","is_repeat_company","company_prior_bookings",
    "company_prior_cancels","company_prior_cancel_rate","company_freq","freq_bucket"})
print(f"base roster: {len(NUMERIC_BASE)} numeric + {len(CATEGORICAL_BASE)} categorical (company held out)")

test_mask = df["is_temporal_test"].astype(bool).to_numpy()
df_tr, df_te = df[~test_mask], df[test_mask]
y = df["y"].to_numpy(); y_tr, y_te = y[~test_mask], y[test_mask]
print(f"rows {len(df):,} | train {len(df_tr):,} | test {len(df_te):,}")

base roster: 14 numeric + 4 categorical (company held out)
rows 169,617 | train 127,212 | test 42,405


In [3]:
# ---- profile: does company even matter? ----
print(f"unique companies (clustered): {comp[comp.str.len()>0].nunique():,}")
print(f"bookings with a company: {df['has_company'].mean():.1%}")
print(f"\ncancel rate by has_company:\n{df.groupby('has_company')['y'].agg(['mean','size'])}")
print(f"\ncancel rate by is_repeat_company:\n{df.groupby('is_repeat_company')['y'].agg(['mean','size'])}")
print(f"\ncancel rate by company frequency bucket:\n{df.groupby('freq_bucket')['y'].agg(['mean','size'])}")
hist = df[df['company_prior_bookings']>0]
if len(hist)>200:
    q = pd.qcut(hist['company_prior_cancel_rate'].fillna(0), 5, duplicates='drop')
    print(f"\ncancel rate by prior_cancel_rate quintile (repeat companies only):\n"
          f"{hist.groupby(q)['y'].agg(['mean','size'])}")

unique companies (clustered): 11,620
bookings with a company: 31.6%

cancel rate by has_company:
                 mean    size
has_company                  
0            0.230777  115965
1            0.156602   53652

cancel rate by is_repeat_company:
                       mean    size
is_repeat_company                  
0                  0.223168  127590
1                  0.159183   42027

cancel rate by company frequency bucket:
                 mean    size
freq_bucket                  
1            0.130643    6690
2-5          0.171188    9878
20+          0.154935   27347
6-20         0.164322    9737
none         0.230777  115965

cancel rate by prior_cancel_rate quintile (repeat companies only):
                               mean   size
company_prior_cancel_rate                 
(-0.001, 0.0485]           0.084587  16811
(0.0485, 0.177]            0.132421   8405
(0.177, 0.247]             0.181083   8405
(0.247, 1.0]               0.313229   8406


In [4]:
# ---- benchmark ----
def build(df_tr, df_te, y_tr, arm):
    num = [c for c in NUMERIC_BASE if c in df_tr.columns]
    cat = [c for c in CATEGORICAL_BASE if c in df_tr.columns]
    ni = SimpleImputer(strategy="median").fit(df_tr[num]); sc = StandardScaler().fit(ni.transform(df_tr[num]))
    Xtr=[sc.transform(ni.transform(df_tr[num]))]; Xte=[sc.transform(ni.transform(df_te[num]))]
    ct=_obj(df_tr[cat]); ce=_obj(df_te[cat])
    ci=SimpleImputer(strategy="most_frequent").fit(ct)
    oh=OneHotEncoder(handle_unknown="ignore",sparse_output=False).fit(ci.transform(ct))
    Xtr.append(oh.transform(ci.transform(ct))); Xte.append(oh.transform(ci.transform(ce)))

    def add_num(cols):
        a=SimpleImputer(strategy="median").fit(df_tr[cols]); s=StandardScaler().fit(a.transform(df_tr[cols]))
        Xtr.append(s.transform(a.transform(df_tr[cols]))); Xte.append(s.transform(a.transform(df_te[cols])))
    if arm=="drop": pass
    elif arm=="has_company":
        Xtr.append(df_tr[["has_company"]].to_numpy(float)); Xte.append(df_te[["has_company"]].to_numpy(float))
    elif arm=="is_repeat":
        Xtr.append(df_tr[["is_repeat_company"]].to_numpy(float)); Xte.append(df_te[["is_repeat_company"]].to_numpy(float))
    elif arm=="freq_bucket":
        oc=OneHotEncoder(handle_unknown="ignore",sparse_output=False).fit(_obj(df_tr[["freq_bucket"]]))
        Xtr.append(oc.transform(_obj(df_tr[["freq_bucket"]]))); Xte.append(oc.transform(_obj(df_te[["freq_bucket"]])))
    elif arm=="history":
        tr=df_tr.assign(log_prior=np.log1p(df_tr["company_prior_bookings"]))
        te=df_te.assign(log_prior=np.log1p(df_te["company_prior_bookings"]))
        cols=["has_company","log_prior","company_prior_cancel_rate"]
        a=SimpleImputer(strategy="constant",fill_value=0.0).fit(tr[cols]); s=StandardScaler().fit(a.transform(tr[cols]))
        Xtr.append(s.transform(a.transform(tr[cols]))); Xte.append(s.transform(a.transform(te[cols])))
    elif arm=="target_oof":
        te_tr,te_te=target_encode_oof(df_tr[COMPANY_COL], y_tr, df_te[COMPANY_COL])
        s=StandardScaler().fit(te_tr); Xtr.append(s.transform(te_tr)); Xte.append(s.transform(te_te))
    return np.hstack(Xtr), np.hstack(Xte)

arms=["drop","has_company","is_repeat","freq_bucket","history","target_oof"]
models={"logreg":lambda:LogisticRegression(solver="saga",C=1.0,max_iter=1000,tol=1e-3),
        "histgb":lambda:HistGradientBoostingClassifier(max_depth=8,learning_rate=0.05,max_iter=400,random_state=42)}
rows=[]
for arm in arms:
    Xtr,Xte=build(df_tr,df_te,y_tr,arm)
    for mn,mf in models.items():
        clf=mf().fit(Xtr,y_tr); p=clf.predict_proba(Xte)[:,1]
        lo,hi=bootstrap_auc_ci(y_te,p)
        rows.append({"arm":arm,"model":mn,"n_features":Xtr.shape[1],
                     "auc":roc_auc_score(y_te,p),"auc_ci_lo":lo,"auc_ci_hi":hi,
                     "ap":average_precision_score(y_te,p)})
        print(f"  {arm:12s} {mn:7s} feats={Xtr.shape[1]:4d} AUC={rows[-1]['auc']:.4f} "
              f"[{lo:.4f},{hi:.4f}] AP={rows[-1]['ap']:.4f}")
res=pd.DataFrame(rows)
from src import tables_dir
out=tables_dir()/"00_audit"/"company_encoding_benchmark.csv"; out.parent.mkdir(parents=True,exist_ok=True)
res.to_csv(out,index=False)
for mn in models:
    print(f"\n[{mn}] sorted by hold-out AUC")
    print(res[res.model==mn].sort_values("auc",ascending=False)
          [["arm","n_features","auc","auc_ci_lo","auc_ci_hi","ap"]]
          .to_string(index=False,float_format=lambda v:f"{v:.4f}"))
print(f"\nsaved -> {out}")

  drop         logreg  feats=  75 AUC=0.7280 [0.7221,0.7340] AP=0.4061
  drop         histgb  feats=  75 AUC=0.7788 [0.7736,0.7842] AP=0.4740
  has_company  logreg  feats=  76 AUC=0.7323 [0.7269,0.7385] AP=0.4095
  has_company  histgb  feats=  76 AUC=0.7813 [0.7761,0.7871] AP=0.4735
  is_repeat    logreg  feats=  76 AUC=0.7290 [0.7231,0.7349] AP=0.4073
  is_repeat    histgb  feats=  76 AUC=0.7787 [0.7735,0.7841] AP=0.4725
  freq_bucket  logreg  feats=  80 AUC=0.7325 [0.7269,0.7387] AP=0.4093
  freq_bucket  histgb  feats=  80 AUC=0.7807 [0.7754,0.7860] AP=0.4747
  history      logreg  feats=  78 AUC=0.7412 [0.7356,0.7470] AP=0.4199
  history      histgb  feats=  78 AUC=0.7882 [0.7830,0.7939] AP=0.4882
  target_oof   logreg  feats=  76 AUC=0.7307 [0.7252,0.7369] AP=0.4052
  target_oof   histgb  feats=  76 AUC=0.7822 [0.7768,0.7879] AP=0.4736

[logreg] sorted by hold-out AUC
        arm  n_features    auc  auc_ci_lo  auc_ci_hi     ap
    history          78 0.7412     0.7356     0.7470 0.

## How to read it

Decision rule applied:
- If every arm sat inside `drop`'s bootstrap CI -> company adds nothing beyond
  `has_corporate_code` / channel / rate plan, and we'd **cut the clustering**.
- If `history` or `target_oof` better than `drop` -> company carries real signal.
- `freq_bucket` uses full-frame frequency (mild leak), so trusted only if it beat
  the leakage-safe `history` by a wide margin.

**Outcome:** `history` cleared `drop` cleanly and `target_oof` underperformed, so
the leakage-safe history bundle is in 00 §5.2 